# 處理美股資料

1. 在 網址： https://www.nasdaq.com/market-activity/stocks/brk.b/historical?page=1&rows_per_page=10&timeline=y5
  下載歷史資料
2. 放到 "history_data/us/stock_price_raw"


# 讀取舊格式的檔案並轉換為新格式

In [4]:
import csv
import os
from datetime import datetime


def convert_file(input_file, output_file):
    with open(input_file, mode='r', encoding='utf-8') as infile:
        reader = csv.DictReader(infile)
        rows = []

        for row in reader:

            date = datetime.strptime(row["Date"], "%m/%d/%Y").strftime("%Y%m%d")

            # replace $ sign and comma
            close_price = row["Close/Last"].replace("$", "")
            open_price = row["Open"].replace("$", "")
            high_price = row["High"].replace("$", "")
            low_price = row["Low"].replace("$", "")

            rows.append({
                "Date": date,
                "Open": open_price,
                "High": high_price,
                "Low": low_price,
                "Close": close_price,
            })
    # reverse
    rows = rows[::-1]
    
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    with open(output_file, mode='w', encoding='utf-8', newline='') as outfile:
        fieldnames = ["Date", "Volume",  "Open", "High", "Low", "Close"]
        writer = csv.DictWriter(outfile, fieldnames=fieldnames)

        writer.writeheader()
        writer.writerows(rows)



# 畫技術分析


In [5]:
import pandas as pd
import numpy as np
import json
import requests
import time
import os


def cal_tech(id):

    path = os.path.join(os.path.abspath(os.path.dirname(os.getcwd())), 'history_data', 'us', 'stock_price')
    # folder_list = ['2330', '2454', '2317', '2382', '2308', '3711']


    stock_tech = pd.read_csv(os.path.join(path , id + '.csv'))

    # # 將stock_tech的值都換成float
    # stock_tech['Open'] = stock_tech['Open'].map(lambda x: float(x.replace(',', '')))
    # stock_tech['Close'] = stock_tech['Close'].map(lambda x: float(x.replace(',', '')))
    # stock_tech['High'] = stock_tech['High'].map(lambda x: float(x.replace(',', '')))
    # stock_tech['Low'] = stock_tech['Low'].map(lambda x: float(x.replace(',', '')))


    # 計算移動均線 (SMA)
    stock_tech['MA5'] = stock_tech['Close'].rolling(window=5).mean()
    stock_tech['MA10'] = stock_tech['Close'].rolling(window=10).mean()
    stock_tech['MA20'] = stock_tech['Close'].rolling(window=20).mean()

    # 計算指數移動平均線 (EMA)
    stock_tech['EMA5'] = stock_tech['Close'].ewm(span=5, adjust=False).mean()
    stock_tech['EMA12'] = stock_tech['Close'].ewm(span=12, adjust=False).mean()
    stock_tech['EMA26'] = stock_tech['Close'].ewm(span=26, adjust=False).mean()

    # 計算MACD和DIF
    stock_tech['DIF'] = stock_tech['EMA12'] - stock_tech['EMA26']
    stock_tech['MACD'] = stock_tech['DIF'].ewm(span=9, adjust=False).mean()


    # 計算RSI
    def calculate_rsi(series, period=14):
        delta = series.diff(1)
        gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
        rs = gain / loss
        rsi = 100 - (100 / (1 + rs))
        return rsi

    stock_tech['RSI'] = calculate_rsi(stock_tech['Close'])

    # 計算KDJ
    low_list = stock_tech['Low'].rolling(9, min_periods=9).min()
    low_list.fillna(value=stock_tech['Low'].expanding().min(), inplace=True)
    high_list = stock_tech['High'].rolling(9, min_periods=9).max()
    high_list.fillna(value=stock_tech['High'].expanding().max(), inplace=True)
    rsv = (stock_tech['Close'] - low_list) / (high_list - low_list) * 100


    # 初始值設定
    stock_tech['K'] = 50
    stock_tech['D'] = 50

    for i in range(1, len(stock_tech)):
        stock_tech.loc[stock_tech.index[i], 'K'] = (2/3) * stock_tech.loc[stock_tech.index[i-1], 'K'] + (1/3) * rsv[i]
        stock_tech.loc[stock_tech.index[i], 'D'] = (2/3) * stock_tech.loc[stock_tech.index[i-1], 'D'] + (1/3) * stock_tech.loc[stock_tech.index[i], 'K']


    # 顯示DataFrame
    stock_tech.to_csv(os.path.join(path , id + 'tech.csv'), encoding='utf-8', index=False)

In [6]:
data = [
    ["SPX", "s&p500", "us"],
]

for d in data:
    # 指定檔案路徑
    stock_id = d[0]
    input_file = os.path.join(os.path.abspath(os.path.dirname(os.getcwd())), 'history_data', 'us', 'stock_price_raw', stock_id + '.csv')
    output_file = os.path.join(os.path.abspath(os.path.dirname(os.getcwd())), 'history_data', 'us', 'stock_price', stock_id + '.csv')
    convert_file(input_file, output_file)
    print(f"檔案已轉換並儲存至 {output_file}")
    cal_tech(stock_id)
    print('save ' + stock_id + 'tech.csv')

檔案已轉換並儲存至 /Users/yanyifu/Documents/_Coding/LLM Predict Stock/history_data/us/stock_price/SPX.csv
save SPXtech.csv


/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_18660/2827683452.py:63: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '60.465418616744415' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  stock_tech.loc[stock_tech.index[i], 'K'] = (2/3) * stock_tech.loc[stock_tech.index[i-1], 'K'] + (1/3) * rsv[i]
/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_18660/2827683452.py:64: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '53.48847287224813' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  stock_tech.loc[stock_tech.index[i], 'D'] = (2/3) * stock_tech.loc[stock_tech.index[i-1], 'D'] + (1/3) * stock_tech.loc[stock_tech.index[i], 'K']
